# 20 — Sequential action components and environment-step returns

Control [Gymnasium Reacher-v5](https://gymnasium.farama.org/environments/mujoco/reacher/),
a two-joint arm reaching toward a target. Choose the first joint torque, then the second
conditioned on the first, and only then advance the environment with `[shoulder_torque, elbow_torque]`.
One shared head emits 33 choices at each of two readout tokens, instead of 33² joint choices.
Both torques lie in `[-1, 1]`; they act on the same arm and jointly determine its motion.
This is a small executable wiring and training example, not a trained reaching agent.

Install the repository's `examples` extra, which includes `gymnasium[mujoco]`.
The example runs on CPU without rendering, so no display or model downloads are needed.

Each component choice is a separate **decision row**, with one readout token. The transformer
sees the earlier choices through causal history. The physical environment advances once
per completed vector; the two clocks are deliberately separate.

Set `BACKUP_MODE` to:
- `"next_action"`: ordinary one-step DQN, bootstrapping from the next component.
- `"n_actions"`: `nstep_gate(n=ACTION_COMPONENTS)`, reaching the same component in the next environment step.
- `"next_step"`: `nstep_gate(n=1, step_key="environment_step")`, reaching the next environment boundary from every component.

Internal rows have reward 0, discount 1, and no done flags. Boundary rows carry Reacher's
reward, physical observations, and discount γ. The marker describes the transition **into**
the row, as do action IDs and rewards. N-step targets use recorded intermediate choices:
they evaluate those sampled completions, not an exact maximum over all possible suffixes.


In [ ]:
from copy import deepcopy

import gymnasium as gym
import numpy as np
from tokenizers import Tokenizer as TextTokenizer
from tokenizers.models import WordLevel
import torch
from torch import nn
from transformers import PreTrainedTokenizerFast

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    environment_episode_ranges,
    DataLoader,
    Datastore,
    EventLog,
    Tokenizer,
    to_device,
)
from mouse_gym import RolloutTracker
from mouse_core.models import (
    action_readouts,
    Model,
    DecodeCache,
    prepare_decode,
    Backbone,
    Embedding,
    MultiEmbedding,
    DiscretizedActionHead,
)
from mouse_core.objectives import DqnObjective, boundary_discount, nstep_gate

# Environment
MAX_ACTIONS = 33
episode_limit = gym.spec("Reacher-v5").max_episode_steps
assert episode_limit is not None
MAX_STEPS_PER_EPISODE = episode_limit  # native limit
BACKUP_MODE = "next_step"

# Model
HIDDEN_DIM = 32
BACKBONE_LAYERS = 1
BACKBONE_HEADS = 2
NUM_FREQUENCIES = 4
HEAD_LAYERS = 1
HEAD_SCALE = 0.1  # final prediction linear's initialization

# Collection and rollout
EPISODES_PER_CYCLE = 2
EPSILON = 0.2

# Training
NUM_CYCLES = 2
TRAIN_STEPS = 2

# Optimizer learning rates
LR_EMBEDDING = 3e-4  # token embeddings and numeric Fourier features
LR_BACKBONE = 3e-4  # backbone
LR_HEADS = 3e-4  # prediction heads and their input RMSNorms

# Target interpolation rates
POLYAK_TAU_EMBEDDING = 0.01  # delayed embedding
POLYAK_TAU_BACKBONE = 0.01  # delayed backbone
POLYAK_TAU_VALUE = 0.01  # delayed value head

# Random seeds
SEED = 7

# Runtime
device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.manual_seed(SEED)
action_rng = np.random.default_rng(SEED)


## Reacher environment

Reacher has one continuous action vector with two joint torques and a ten-value
observation (joint angles encoded as sine/cosine, target position, joint velocities,
and fingertip displacement). Both torque selections happen before `env.step`;
internal choices cannot reveal a future physical observation. The environment supplies
one reward for the complete vector.

Environment payloads split the vector into shoulder/elbow cosine, sine, and angular
velocity; target x/y; and fingertip x/y offsets from the target. The ordered
`OBSERVATION_FIELDS` names below match Gymnasium’s observation vector.

Use Reacher-v5's native limit of 50 physical steps (100 component decisions). The
environment rewards fingertip proximity throughout the episode and truncates at that
limit; reaching the target does not terminate it. This example learns the discounted return
within that complete 50-step episode, so the time limit ends the training horizon.
Replay sizing reads the native limit instead of shortening it.


In [ ]:
env = gym.make("Reacher-v5")
assert isinstance(env.action_space, gym.spaces.Box)
assert isinstance(env.observation_space, gym.spaces.Box)
assert env.action_space.shape == (2,)
assert env.observation_space.shape == (10,)
ACTION_COMPONENTS = env.action_space.shape[0]
OBSERVATION_DIM = env.observation_space.shape[0]
ACTION_FIELDS = ("shoulder_torque", "elbow_torque")
# Reacher-v5 observation order; offsets are fingertip minus target position.
OBSERVATION_FIELDS = (
    "shoulder_cos", "elbow_cos", "shoulder_sin", "elbow_sin",
    "target_x", "target_y", "shoulder_velocity", "elbow_velocity",
    "fingertip_offset_x", "fingertip_offset_y",
)
assert len(OBSERVATION_FIELDS) == OBSERVATION_DIM
assert len(ACTION_FIELDS) == ACTION_COMPONENTS
# Both joints share the same bounds, allowing one scalar codebook and Q head.
np.testing.assert_array_equal(env.action_space.low, [-1.0, -1.0])
np.testing.assert_array_equal(env.action_space.high, [1.0, 1.0])
action_vectors = torch.linspace(-1.0, 1.0, MAX_ACTIONS)[:, None]
assert all(env.action_space.contains(np.repeat(v.numpy(), ACTION_COMPONENTS))
           for v in action_vectors)


## Tokenization

`component` identifies the torque to choose next. Agent payloads record either
`shoulder_torque` or `elbow_torque` for the selected joint;
all physical observations remain unchanged between choices. Each row has one readout.
The category ID is only an objective label, while its physical torque enters the model.
The final observation retains the last reward and boundary flags; reset episodes have separate groups.


In [ ]:
def on_episode_start(ctx):
    return ctx.get("episode_start", False)

vocabulary = {"[UNK]": 0, "reward": 1, "reset": 2, "readout": 3, "component": 4}
for field in (*ACTION_FIELDS, *OBSERVATION_FIELDS):
    vocabulary[field] = len(vocabulary)
text_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=TextTokenizer(WordLevel(vocab=vocabulary, unk_token="[UNK]")),
    unk_token="[UNK]",
)

tokenizer = Tokenizer(
    step_field="step_index",
    tokenizers={"text": text_tokenizer},
    input_fields=[
        {
            'type': 'numeric',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'component',
            'format': 'component',
            'fourier_min': -1.0,
            'fourier_max': float(ACTION_COMPONENTS + 1),
        },
        {
            'type': 'token',
            "tokenizer": "text",
            'provider': 'coordinator',
            'format': 'reset',
            'when': on_episode_start,
        },
        *[{'type': 'numeric', 'provider': None, 'input_field': field, 'format': field, 'fourier_min': -2.0, 'fourier_max': 2.0} for field in ACTION_FIELDS],
        *[{'type': 'numeric', 'provider': None, 'input_field': field, 'format': field, 'fourier_min': -16.0, 'fourier_max': 16.0} for field in OBSERVATION_FIELDS],
        {
            'type': 'numeric',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'reward',
            'format': 'reward',
            'fourier_min': -32.0,
            'fourier_max': 32.0,
        },
        {
            'type': 'token',
            "tokenizer": "text",
            'provider': None,
            'input_field': 'action_requested',
            'format': '\n',
            'head_output': True,
        },
    ],
    objective_fields=[
        {"input_field": "action",
        "missing_value": None,
    },
        {"input_field": "reward",
        "missing_value": 0.0,
    },
        {"input_field": "terminated",
        "missing_value": False,
    },
        {"input_field": "truncated",
        "missing_value": False,
    },
        {"input_field": "environment_step"},
    ],
)


## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference
uses the same encoder and returns cache and readout-mask metadata. A `ModuleDict`
called `modules` holds references to the online components for checkpointing and
mode changes; it is not called for predictions.

The delayed encoder is a frozen copy. Separate backup value/action heads track
the online Q head(s) with independent Polyak rates. Only online parameters enter
the optimizer.

The Q head owns the fixed action-vector table and its continuous/discrete
conversion methods. The action table is saved as part of the Q head.



In [ ]:
class Encoder(Model):
    def __init__(self, *, hidden_dim, vocab_size, num_frequencies, num_layers, num_heads,
                 train_kernel, decode_kernel, dtype):
        super().__init__()
        self.embedding = MultiEmbedding(embeddings={
            "text": Embedding(hidden_dim=hidden_dim, vocab_size=vocab_size,
                              num_frequencies=num_frequencies),
        })
        self.backbone = Backbone(
            architecture="qwen3", hidden_dim=hidden_dim, num_layers=num_layers,
            num_heads=num_heads, train_kernel=train_kernel,
            decode_kernel=decode_kernel, dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to_tensors(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_ids"]
        head_step_ids = tensors["head_step_ids"]
        embed = self.embedding(
            token_ids, numeric_values=numeric_values, numeric_mask=numeric_mask,
            tokenizer_ids=tensors["tokenizer_ids"], tokenizer_names=tokens.tokenizer_names,
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed, tokens=tokens, sequence_ids=sequence_ids,
                head_token_ids=head_token_ids, head_step_ids=head_step_ids,
            )
            if cache is None:
                cache = DecodeCache(sessions=(self.backbone.decode_session(batch_size=tokens.B),))
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape, features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }

class QHead(DiscretizedActionHead):
    def __init__(self, *, hidden_dim, action_vectors, num_layers, scale):
        super().__init__(
            in_features=hidden_dim, action_vectors=action_vectors,
            hidden_dim=hidden_dim, num_layers=num_layers, scale=scale,
            propagate_gradient=1.0,
        )

encoder = Encoder(
    hidden_dim=HIDDEN_DIM, vocab_size=len(text_tokenizer), num_frequencies=NUM_FREQUENCIES,
    num_layers=BACKBONE_LAYERS, num_heads=BACKBONE_HEADS,
    train_kernel="reference", decode_kernel="flex", dtype=torch.float32,
).train().to(device)

q_head = QHead(
    hidden_dim=encoder.backbone.hidden_dim, action_vectors=action_vectors,
    num_layers=HEAD_LAYERS, scale=HEAD_SCALE,
).train().to(device)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()

head = q_head
print(f"{ACTION_COMPONENTS} sequential choices of {MAX_ACTIONS} torques")


## Collection

Internal decision rows keep the observation fixed; the completed vector advances Reacher once.

The collector appends each category and torque before the resulting environment output.
`RolloutTracker` tracks episode and decision indices from each completed output; its flags control the loop.
The raw event log supplies the live history, and replay reads the same source events.


In [ ]:
event_log = EventLog()
tracker = RolloutTracker()

@torch.no_grad()
def collect_episode() -> tuple[float, list[dict]]:
    modules.eval()
    event_start = len(event_log)
    episode_start = tracker.output is None or tracker.ended
    episode_index = tracker.next_episode_index
    coordinator_event = event_log.event(provider='coordinator', episode_index=episode_index, step_index=(0 if episode_start else tracker.step_index + 1), payload={
        **({"episode_start": True} if episode_start else {}),
        'component': 0,
        'environment_step': False,
    })
    store.append(data=coordinator_event)
    observation, info = env.reset(seed=SEED + tracker.episode_index + 1)
    tracker.update(output=(observation, 0.0, False, False, info))
    environment_event = event_log.event(provider='environment', episode_index=tracker.episode_index, step_index=tracker.step_index, payload={
        **dict(zip(OBSERVATION_FIELDS, tracker.observation, strict=True)),
        **({"reward": tracker.reward} if tracker.reward != 0 else {}),
        **({"terminated": True} if tracker.terminated else {}),
        **({"truncated": True} if tracker.truncated else {}),
        **({"action_requested": True} if tracker.action_requested else {}),
    })
    store.append(data=environment_event)
    history = event_log[event_start:]
    while not tracker.ended:
        torques = []
        for component in range(ACTION_COMPONENTS):
            advanced = component == ACTION_COMPONENTS - 1
            event_start = len(event_log)
            episode_start = tracker.output is None or tracker.ended
            episode_index = tracker.next_episode_index
            coordinator_event = event_log.event(provider='coordinator', episode_index=episode_index, step_index=(0 if episode_start else tracker.step_index + 1), payload={
                **({"episode_start": True} if episode_start else {}),
                'component': (component + 1) % ACTION_COMPONENTS,
                'environment_step': advanced,
            })
            store.append(data=coordinator_event)
            inputs = tokenizer.pack_rows(rows=[history], continuing=None)
            out = encoder(inputs)
            scores = action_readouts(q_head(out["z"]), inputs)[0]
            if action_rng.random() < EPSILON:
                action_id = int(action_rng.integers(MAX_ACTIONS))
            else:
                action_id = int(scores.argmax().item())
            torque = float(head.discrete_to_continuous(action_id)[0])
            agent_event = event_log.event(provider='agent', episode_index=episode_index, step_index=tracker.step_index + 1, payload={
                'action': action_id,
                ACTION_FIELDS[component]: torque,
            })
            store.append(data=agent_event)
            torques.append(torque)
            if advanced:
                continuous_action = np.asarray(torques, dtype=np.float32)
                assert env.action_space.contains(continuous_action)
                observation, reward, terminated, truncated, info = env.step(continuous_action)
                output = (observation, float(reward), terminated, truncated, info)
            else:
                output = (tracker.observation, 0.0, False, False, tracker.info)
            tracker.update(output=output)
            environment_event = event_log.event(provider='environment', episode_index=tracker.episode_index, step_index=tracker.step_index, payload={
                **dict(zip(OBSERVATION_FIELDS, tracker.observation, strict=True)),
                **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                **({"terminated": True} if tracker.terminated else {}),
                **({"truncated": True} if tracker.truncated else {}),
                **({"action_requested": True} if tracker.action_requested else {}),
            })
            store.append(data=environment_event)
            history.extend(event_log[event_start:])
    return tracker.episodes[-1].reward, history


## Choose the backup horizon and train

Only environment transitions apply γ. Both true termination and the native time limit end
this finite-horizon return (`gamma_terminated=0`, `gamma_truncated=0`), retaining the final
reward. The log preserves the environment's actual flags. The final row has no outgoing
training transition. Replay samples complete episodes, and `cross_group_backups="fault"`
raises if a target would require data beyond a sampled group.

With `next_step`, both components get the environment reward and bootstrap from the next
state's first readout until the episode ends. With `next_action`, the first component targets the second
component's value with no reward or discount. With `n_actions`, each target skips
exactly `ACTION_COMPONENTS` decisions, stopping at the episode boundary.


In [ ]:
gates = {
    "next_action": None,
    "n_actions": nstep_gate(n=ACTION_COMPONENTS),
    "next_step": nstep_gate(n=1, step_key="environment_step"),
}
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=0.99, gamma_terminated=0.0, gamma_truncated=0.0,
        step_key="environment_step",
    ),
    temperature=0.0,
    cross_group_backups="fault",
    gate=gates[BACKUP_MODE],
)
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(lr={'embedding': LR_EMBEDDING, 'backbone': LR_BACKBONE}),
        {'params': q_head.parameters(), 'lr': LR_HEADS},
    ],
    lr=LR_BACKBONE,
    weight_decay=0.0,
)

def run_train(*, cycle: int, num_steps: int) -> tuple[float, dict[str, float | torch.Tensor]]:
    if num_steps < 1:
        raise ValueError("num_steps must be positive")
    loader = DataLoader(
        stores=[store],
        events_budget=(3 * MAX_STEPS_PER_EPISODE * ACTION_COMPONENTS + 2) * EPISODES_PER_CYCLE,
        batch_size=1,
        sample_ranges=environment_episode_ranges,
        transform=tokenizer.encode,
        num_workers=0,
        seed=SEED + cycle,
    )
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    try:
        modules.train()
        for _ in range(num_steps):
            inputs, objective_data, group_id = loader.next_batch()
            out = encoder(inputs)
            with torch.no_grad():
                delayed_out = delayed_encoder(inputs)
                backup_action = q_head(delayed_out['z'])
            losses, metrics = objective(
                objective_data=to_device(data=objective_data, device=device),
                group_id=group_id.to(device),
                predictions=q_head(out['z']),
                backup_value_predictions=backup_value_head(delayed_out['z']),
                backup_action_predictions=backup_action,
            )
            loss = torch.stack(tuple(losses.values())).sum()
            assert torch.isfinite(loss)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modules.parameters(), float("inf"), error_if_nonfinite=True)
            optimizer.step()
            polyak_update(
                target=delayed_encoder.embedding,
                source=encoder.embedding,
                tau=POLYAK_TAU_EMBEDDING,
            )
            polyak_update(target=delayed_encoder.backbone, source=encoder.backbone, tau=POLYAK_TAU_BACKBONE)
            polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
        assert loss is not None
        loss_value = float(loss.detach())
        return (loss_value, metrics)
    finally:
        loader.close()


## Run

Collect complete episodes and train from replay.


In [ ]:
store = Datastore(name="sequential-reacher-replay")
training_log = []
last_history: list[dict] | None = None
try:
    for cycle in range(NUM_CYCLES):
        returns = []
        for episode in range(EPISODES_PER_CYCLE):
            episode_return, last_history = collect_episode()
            returns.append(episode_return)
        loss, metrics = run_train(cycle=cycle, num_steps=TRAIN_STEPS)
        training_log.append({'cycle': cycle, 'mean_return': float(np.mean(returns)), 'loss': loss})
        print(
            f"cycle={cycle} | mean return={np.mean(returns):.1f} | loss={loss:.4f} | mean Q={float(metrics['q_values_mean']):.3f} | replay rows={len(store)}",
        )
finally:
    env.close()

assert last_history is not None, "Collect at least one episode before inspecting its history"
print(f"Finished {NUM_CYCLES * EPISODES_PER_CYCLE} episodes and {NUM_CYCLES * TRAIN_STEPS} updates.")


## Check the decision and environment clocks

There are K decisions per physical transition and one final observation row. Changing an
objective-only ID never changes the tokens; later choices instead see the recorded torque.
The checks below verify the recorded torque vector and the two clocks. Each run trains with the selected backup setting.


In [ ]:
coordinators = [event for event in last_history if event["provider"] == "coordinator"]
agents = [event for event in last_history if event["provider"] == "agent"]
results = [event for event in last_history if event["provider"] == "environment"]
assert len(results) == MAX_STEPS_PER_EPISODE * ACTION_COMPONENTS + 1
for index, (control, agent, result) in enumerate(zip(coordinators[1:], agents, results[1:], strict=True), start=1):
    assert control["payload"]["environment_step"] == (index % ACTION_COMPONENTS == 0)
    if not control["payload"]["environment_step"]:
        assert not {"reward", "terminated", "truncated"} & result["payload"].keys()
        assert all(result["payload"][field] == results[index - 1]["payload"][field] for field in OBSERVATION_FIELDS)
    action_field = ACTION_FIELDS[(index - 1) % ACTION_COMPONENTS]
    assert agent["payload"][action_field] == float(head.discrete_to_continuous(agent["payload"]["action"])[0])
print(f"Verified {len(agents)} decisions and "
      f"{sum(event['payload']['environment_step'] for event in coordinators)} environment transitions.")
